# TruthLens — Item 1 pilot (feasibility test)

Short pilot of **M0 (MobileNetV3-Small, true labels only)** on **Tiny-GenImage**, run before the full GenImage subset is ready.

It records the numbers the Item 1 *Feasibility* section asks for:

| Measurement | Where |
|---|---|
| GPU model, time per epoch, peak GPU memory | Training cell |
| Validation accuracy / macro-F1 / AUC | Training cell |
| Parameters, MACs | Model cell |
| CPU latency (median + IQR, batch 1, 1 thread) | Latency cell |
| ONNX export + INT8 quantization works? Size, latency, accuracy after INT8 | Quantization cell |
| Overfit-one-batch test (needed again in Item 2) | Sanity cell |

**Run:** Colab or Kaggle with a GPU (Colab: *Runtime → Change runtime type → T4 GPU*; Kaggle: *Accelerator → GPU T4*, Internet on). Then *Run all*. About 20–40 min, most of it the dataset download (~8 GB) and the first epoch.

**Honesty rules for the proposal:** these are *pilot* numbers on Tiny-GenImage's validation split, 2 epochs, 1 seed. Report them as such. GenImage's test set is never touched here.

In [1]:
!pip -q install datasets onnx onnxruntime thop

In [2]:
import os, io, time, json, random
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from PIL import Image
from datasets import load_dataset
from sklearn.metrics import f1_score, roc_auc_score

# ---- pilot settings (change here) ----
SEED    = 42
N_TRAIN = 10_000   # balanced pilot subset of Tiny-GenImage train (28k available)
N_VAL   = 2_000    # balanced subset of Tiny-GenImage validation (7k available)
EPOCHS  = 2
BATCH   = 64
IMG     = 224
LR      = 3e-4
WORKERS = 2

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if device == "cuda" else "none"
print("Device:", device, "|", GPU, "| torch", torch.__version__)

Device: cuda | Tesla T4 | torch 2.11.0+cu130


## 1. Data — Tiny-GenImage, balanced real/fake across the 8 generators

In [3]:
ds = load_dataset("TheKernel01/Tiny-GenImage")
print(ds)
GEN_NAMES = ds["train"].features["generator"].names   # 0 = Real, 1..8 = generators
print(GEN_NAMES)

def balanced_indices(split, n, seed):
    """n/2 real + n/2 fake, the fake half split equally over the 8 generators."""
    gens = np.array(split["generator"])
    rng = np.random.default_rng(seed)
    real = np.where(gens == 0)[0]
    idx = list(rng.choice(real, size=min(n // 2, len(real)), replace=False))
    per_gen = (n // 2) // 8
    for g in range(1, 9):
        pool = np.where(gens == g)[0]
        idx += list(rng.choice(pool, size=min(per_gen, len(pool)), replace=False))
    rng.shuffle(idx)
    return [int(i) for i in idx]

train_idx = balanced_indices(ds["train"], N_TRAIN, SEED)
val_idx   = balanced_indices(ds["validation"], N_VAL, SEED)
print("train:", len(train_idx), "| val:", len(val_idx))

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'generator'],
        num_rows: 28000
    })
    validation: Dataset({
        features: ['image', 'label', 'generator'],
        num_rows: 7000
    })
})
['Real', 'ADM', 'BigGAN', 'GLIDE', 'Midjourney', 'SD14', 'SD15', 'VQDM', 'Wukong']
train: 9375 | val: 1875


In [4]:
class RandomJPEG:
    """Re-encode every image as JPEG at a random quality, real and fake alike,
    so the model cannot use 'JPEG vs PNG' as a shortcut."""
    def __init__(self, qmin=65, qmax=100): self.qmin, self.qmax = qmin, qmax
    def __call__(self, img):
        q = random.randint(self.qmin, self.qmax)
        buf = io.BytesIO(); img.convert("RGB").save(buf, "JPEG", quality=q); buf.seek(0)
        return Image.open(buf).convert("RGB")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(IMG), transforms.RandomHorizontalFlip(),
    RandomJPEG(65, 100), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG),
    RandomJPEG(95, 95), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class HFImages(Dataset):
    def __init__(self, split, indices, tf): self.split, self.idx, self.tf = split, indices, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        ex = self.split[self.idx[i]]
        return self.tf(ex["image"].convert("RGB")), int(ex["generator"])   # 9-class label

train_loader = DataLoader(HFImages(ds["train"], train_idx, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(HFImages(ds["validation"], val_idx, eval_tf), batch_size=BATCH,
                          shuffle=False, num_workers=WORKERS, pin_memory=True)
# calibration images for INT8 come from TRAIN (no augmentation), never from validation/test
calib_loader = DataLoader(HFImages(ds["train"], train_idx[:640], eval_tf), batch_size=64,
                          shuffle=False, num_workers=WORKERS)

## 2. Model — M0 = MobileNetV3-Small (ImageNet-pretrained), 9-class head

In [5]:
def make_student(num_classes=9):
    m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    return m

from thop import profile
_m = make_student().eval()
N_PARAMS = sum(p.numel() for p in _m.parameters())
MACS, _ = profile(_m, inputs=(torch.randn(1, 3, IMG, IMG),), verbose=False)
print(f"Parameters: {N_PARAMS/1e6:.2f} M | MACs @ {IMG}px: {MACS/1e6:.1f} M (thop: conv + linear layers)")
del _m

Parameters: 1.53 M | MACs @ 224px: 61.5 M (thop: conv + linear layers)


## 3. Sanity check — overfit one batch (loss should go to ~0, accuracy to 1.0)

In [6]:
xb, yb = next(iter(train_loader))
xb, yb = xb[:32].to(device), yb[:32].to(device)
m = make_student().to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
for step in range(100):
    loss = F.cross_entropy(m(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 20 == 0 or step == 99: print(f"step {step:3d}  loss {loss.item():.4f}")
m.eval()
with torch.no_grad(): OVERFIT_ACC = (m(xb).argmax(1) == yb).float().mean().item()
print("Overfit-one-batch accuracy:", OVERFIT_ACC)
del m, opt

step   0  loss 2.2055
step  20  loss 0.0001
step  40  loss 0.0000
step  60  loss 0.0000
step  80  loss 0.0000
step  99  loss 0.0000
Overfit-one-batch accuracy: 1.0


## 4. Train M0 for a short pilot run

In [7]:
def metrics_from_probs(P, Y):
    """P: (N,9) probabilities, Y: (N,) 9-class labels. Binary view: P(AI) = 1 - P(Real)."""
    y_bin = (Y > 0).astype(int)
    p_ai = 1.0 - P[:, 0]
    pred = (p_ai >= 0.5).astype(int)
    return {"acc_binary": float((pred == y_bin).mean()),
            "f1_macro_binary": float(f1_score(y_bin, pred, average="macro")),
            "auc": float(roc_auc_score(y_bin, p_ai)),
            "acc_9class": float((P.argmax(1) == Y).mean())}

@torch.no_grad()
def evaluate(m, loader, dev):
    m.eval(); P, Y, loss_sum, n = [], [], 0.0, 0
    for x, y in loader:
        logits = m(x.to(dev, non_blocking=True)).float()
        loss_sum += F.cross_entropy(logits, y.to(dev), reduction="sum").item(); n += len(y)
        P.append(F.softmax(logits, 1).cpu().numpy()); Y.append(y.numpy())
    out = metrics_from_probs(np.concatenate(P), np.concatenate(Y))
    out["val_loss"] = loss_sum / n
    return out

model = make_student().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
if use_amp: torch.cuda.reset_peak_memory_stats()

history = []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); loss_sum, n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        loss_sum += loss.item() * len(y); n += len(y)
    epoch_time = time.time() - t0
    val = evaluate(model, val_loader, device)
    history.append({"epoch": epoch, "train_loss": loss_sum / n, "epoch_time_s": epoch_time, **val})
    print(f"epoch {epoch}: train_loss {loss_sum/n:.4f} | val_loss {val['val_loss']:.4f} | "
          f"val acc {val['acc_binary']:.4f} | F1 {val['f1_macro_binary']:.4f} | AUC {val['auc']:.4f} | "
          f"9-class acc {val['acc_9class']:.4f} | {epoch_time/60:.1f} min")

PEAK_GPU_GB = torch.cuda.max_memory_allocated() / 1e9 if use_amp else None
print("Peak GPU memory (GB):", PEAK_GPU_GB)
torch.save(model.state_dict(), "m0_pilot.pt")

epoch 1: train_loss 1.1766 | val_loss 0.8244 | val acc 0.8139 | F1 0.8110 | AUC 0.8849 | 9-class acc 0.7056 | 1.4 min
epoch 2: train_loss 0.7296 | val_loss 0.7484 | val acc 0.8027 | F1 0.8026 | AUC 0.8977 | 9-class acc 0.7163 | 1.2 min
Peak GPU memory (GB): 0.632549888


## 5. CPU latency (PyTorch FP32) — batch 1, 1 thread, warm-up, median + IQR

In [8]:
def latency_ms(fn, warmup=20, runs=200):
    for _ in range(warmup): fn()
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); fn(); ts.append((time.perf_counter() - t) * 1000)
    q25, q50, q75 = np.percentile(ts, [25, 50, 75])
    return float(q50), float(q75 - q25)

torch.set_num_threads(1)
cpu_model = make_student()
cpu_model.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})
cpu_model.eval()
x1 = torch.randn(1, 3, IMG, IMG)
with torch.inference_mode():
    TORCH_MED, TORCH_IQR = latency_ms(lambda: cpu_model(x1))
CPU_NAME = os.popen("lscpu | grep 'Model name' | head -1").read().split(":")[-1].strip()
print(f"PyTorch FP32 CPU latency: {TORCH_MED:.2f} ms (IQR {TORCH_IQR:.2f}) on {CPU_NAME}")

PyTorch FP32 CPU latency: 9.03 ms (IQR 0.46) on Intel(R) Xeon(R) CPU @ 2.00GHz


## 6. Early quantization test — ONNX export → INT8 (ONNX Runtime static quantization)

This is the risk check from the plan: does MobileNetV3-Small export and quantize cleanly, and does INT8 actually get faster *on this CPU*? (Speed-ups depend on the CPU's integer instructions — a small or zero gain here is a real finding, not a bug.)

In [9]:
!pip install -q onnx onnxscript onnxruntime

In [10]:
# ===== REPLACEMENT for section 6 (quantization) — paste over the old cell and run =====
# Fixes: (1) FP32 file size was wrong (new exporter saves weights in a separate .data file)
#        (2) INT8 accuracy collapsed to chance -> try safer quantization settings
import onnx, onnxruntime as ort
from onnxruntime.quantization import (quantize_static, CalibrationDataReader, QuantFormat,
                                      QuantType, CalibrationMethod)
from onnxruntime.quantization.shape_inference import quant_pre_process

# 1) Export with ALL weights inside one .onnx file
try:   # classic exporter: simpler graph, quantizes more reliably
    torch.onnx.export(cpu_model, x1, "m0_fp32.onnx", input_names=["input"], output_names=["logits"],
                      opset_version=17, dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
                      dynamo=False)
    print("exported with classic exporter (opset 17)")
except Exception as e:
    print("classic exporter unavailable -> new exporter:", type(e).__name__)
    torch.onnx.export(cpu_model, (x1,), "m0_fp32.onnx", input_names=["input"], output_names=["logits"],
                      opset_version=18, dynamo=True, external_data=False,
                      dynamic_shapes={"x": {0: torch.export.Dim("batch")}})
onnx.checker.check_model("m0_fp32.onnx")
quant_pre_process("m0_fp32.onnx", "m0_prep.onnx")

def size_mb(path):
    extra = path + ".data"
    return (os.path.getsize(path) + (os.path.getsize(extra) if os.path.exists(extra) else 0)) / 1e6

class TrainCalib(CalibrationDataReader):          # calibration images from TRAIN only
    def __init__(self, loader): self.it = iter(loader)
    def get_next(self):
        try: x, _ = next(self.it)
        except StopIteration: return None
        return {"input": x.numpy()}

def session(path):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1; so.inter_op_num_threads = 1
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

def eval_onnx(sess, loader):
    P, Y = [], []
    for x, y in loader:
        logits = sess.run(None, {"input": x.numpy()})[0]
        P.append(torch.softmax(torch.from_numpy(logits), 1).numpy()); Y.append(y.numpy())
    return metrics_from_probs(np.concatenate(P), np.concatenate(Y))

# 2) Three INT8 settings, from "quantize everything" to "only conv/linear layers"
configs = {
    "INT8-A all ops, MinMax, QDQ":         dict(quant_format=QuantFormat.QDQ, calibrate_method=CalibrationMethod.MinMax),
    "INT8-B conv+linear, Percentile, QDQ": dict(quant_format=QuantFormat.QDQ, calibrate_method=CalibrationMethod.Percentile,
                                                op_types_to_quantize=["Conv", "Gemm", "MatMul"]),
    "INT8-C conv+linear, Percentile, QOp": dict(quant_format=QuantFormat.QOperator, calibrate_method=CalibrationMethod.Percentile,
                                                op_types_to_quantize=["Conv", "Gemm", "MatMul"]),
}
paths = {"FP32 (ONNX)": "m0_fp32.onnx"}
for i, (name, cfg) in enumerate(configs.items()):
    out = f"m0_int8_{i}.onnx"
    try:
        quantize_static("m0_prep.onnx", out, TrainCalib(calib_loader), per_channel=True,
                        weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8, **cfg)
        paths[name] = out
    except Exception as e:
        print(f"{name}: quantization failed -> {type(e).__name__}: {e}")

results = {}
xin = x1.numpy()
for name, path in paths.items():
    s = session(path)
    med, iqr = latency_ms(lambda: s.run(None, {"input": xin}))
    acc = eval_onnx(s, val_loader)
    results[name] = {"size_MB": size_mb(path), "latency_ms": med, "latency_IQR_ms": iqr, **acc}
    print(f"{name:38s} {results[name]['size_MB']:5.2f} MB | {med:5.2f} ms (IQR {iqr:.2f}) | "
          f"val acc {acc['acc_binary']:.4f} | AUC {acc['auc']:.4f}")

/tmp/ipykernel_11684/4002628367.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(cpu_model, x1, "m0_fp32.onnx", input_names=["input"], output_names=["logits"],


exported with classic exporter (opset 17)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 103
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 103
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
FP32 (ONNX)                            12.20 MB |  2.20 ms (IQR 0.16) | val acc 0.8027 | AUC 0.8977
INT8-A all ops, MinMax, QDQ             1.88 MB |  3.97 ms (IQR 0.31) | val acc 0.4667 | AUC 0.4743
INT8-B conv+linear, Percentile, QDQ     1.86 MB |  4.56 ms (IQR 0.50) | val acc 0.4731 | AUC 0.5310
INT8-C conv+linear, Percentile, QOp     1.73 MB |  4.84 ms (IQR 0.52) | val acc 0.4683 | AUC 0.5297


In [11]:
# ===== Quantization test 2 — run as a NEW cell right after the quantization cell =====
# Checks one likely cause of the collapse: on Intel CPUs WITHOUT "VNNI" instructions,
# ONNX Runtime's default INT8 setting can overflow ("saturate") inside conv layers.
# Fixes to try: reduce_range=True, or 8-bit unsigned weights (U8U8), or per-tensor scales.
import subprocess
flags = subprocess.run("lscpu", capture_output=True, text=True).stdout.lower()
print("CPU has VNNI:", any(f in flags for f in ["avx512_vnni", "avx512vnni", "avx_vnni"]))

configs2 = {
    "INT8-D reduce_range (U8S8)": dict(weight_type=QuantType.QInt8, extra={"reduce_range": True}),
    "INT8-E U8U8 weights":        dict(weight_type=QuantType.QUInt8, extra={}),
    "INT8-F per-tensor, reduce":  dict(weight_type=QuantType.QInt8, extra={"reduce_range": True}, per_tensor=True),
}
for i, (name, cfg) in enumerate(configs2.items()):
    out = f"m0_int8_b{i}.onnx"
    try:
        quantize_static("m0_prep.onnx", out, TrainCalib(calib_loader),
                        quant_format=QuantFormat.QDQ,
                        per_channel=not cfg.get("per_tensor", False),
                        calibrate_method=CalibrationMethod.Percentile,
                        op_types_to_quantize=["Conv", "Gemm", "MatMul"],
                        weight_type=cfg["weight_type"], activation_type=QuantType.QUInt8,
                        **cfg["extra"])
    except Exception as e:
        print(f"{name}: failed -> {type(e).__name__}: {e}"); continue
    s = session(out)
    med, iqr = latency_ms(lambda: s.run(None, {"input": xin}))
    acc = eval_onnx(s, val_loader)
    results[name] = {"size_MB": size_mb(out), "latency_ms": med, "latency_IQR_ms": iqr, **acc}
    print(f"{name:38s} {results[name]['size_MB']:5.2f} MB | {med:5.2f} ms (IQR {iqr:.2f}) | "
          f"val acc {acc['acc_binary']:.4f} | AUC {acc['auc']:.4f}")

# Sanity check: does INT8 output the same class for every image? (sign of a broken model)
s_int8 = session("m0_int8_1.onnx") if os.path.exists("m0_int8_1.onnx") else None
if s_int8:
    xb_np = next(iter(val_loader))[0].numpy()
    fp = session("m0_fp32.onnx").run(None, {"input": xb_np})[0].argmax(1)
    q  = s_int8.run(None, {"input": xb_np})[0].argmax(1)
    print("FP32 predicted classes (first 20):", fp[:20])
    print("INT8-B predicted classes (first 20):", q[:20])

CPU has VNNI: False
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 103
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
INT8-D reduce_range (U8S8)              1.86 MB |  4.52 ms (IQR 0.36) | val acc 0.4939 | AUC 0.5228
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 103
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


INT8-E U8U8 weights                     1.87 MB |  5.71 ms (IQR 0.46) | val acc 0.4795 | AUC 0.5379
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 103
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
INT8-F per-tensor, reduce               1.73 MB |  4.62 ms (IQR 0.60) | val acc 0.4667 | AUC 0.5350
FP32 predicted classes (first 20): [0 2 4 0 3 4 0 7 0 4 4 0 2 8 6 3 0 3 6 0]
INT8-B predicted classes (first 20): [8 6 8 8 0 8 8 8 8 8 8 8 6 8 8 3 8 0 8 8]


## 7. Pilot summary — copy into the Feasibility section (label as pilot results)

In [12]:
summary = {
    "dataset": f"Tiny-GenImage pilot: {len(train_idx)} train / {len(val_idx)} val (balanced), {IMG}px, JPEG-aug",
    "gpu": GPU, "cpu": CPU_NAME, "epochs": EPOCHS, "batch": BATCH, "seed": SEED,
    "params_M": round(N_PARAMS / 1e6, 2), "MACs_M": round(MACS / 1e6, 1),
    "overfit_one_batch_acc": OVERFIT_ACC,
    "time_per_epoch_min": round(history[-1]["epoch_time_s"] / 60, 2),
    "peak_gpu_mem_GB": round(PEAK_GPU_GB, 2) if PEAK_GPU_GB else None,
    "val_acc_binary": round(history[-1]["acc_binary"], 4),
    "val_f1_macro": round(history[-1]["f1_macro_binary"], 4),
    "val_auc": round(history[-1]["auc"], 4),
    "torch_fp32_cpu_latency_ms": round(TORCH_MED, 2),
    "onnx": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in results.items()},
    "history": history,
}
print(json.dumps({k: v for k, v in summary.items() if k != "history"}, indent=2))
with open("pilot_results.json", "w") as f: json.dump(summary, f, indent=2)
print("Saved pilot_results.json — commit it to GitHub (results/).")

{
  "dataset": "Tiny-GenImage pilot: 9375 train / 1875 val (balanced), 224px, JPEG-aug",
  "gpu": "Tesla T4",
  "cpu": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "epochs": 2,
  "batch": 64,
  "seed": 42,
  "params_M": 1.53,
  "MACs_M": 61.5,
  "overfit_one_batch_acc": 1.0,
  "time_per_epoch_min": 1.22,
  "peak_gpu_mem_GB": 0.63,
  "val_acc_binary": 0.8027,
  "val_f1_macro": 0.8026,
  "val_auc": 0.8977,
  "torch_fp32_cpu_latency_ms": 9.03,
  "onnx": {
    "FP32 (ONNX)": {
      "size_MB": 12.1998,
      "latency_ms": 2.2002,
      "latency_IQR_ms": 0.1567,
      "acc_binary": 0.8027,
      "f1_macro_binary": 0.8026,
      "auc": 0.8977,
      "acc_9class": 0.7163
    },
    "INT8-A all ops, MinMax, QDQ": {
      "size_MB": 1.8762,
      "latency_ms": 3.9744,
      "latency_IQR_ms": 0.3107,
      "acc_binary": 0.4667,
      "f1_macro_binary": 0.3182,
      "auc": 0.4743,
      "acc_9class": 0.0672
    },
    "INT8-B conv+linear, Percentile, QDQ": {
      "size_MB": 1.8585,
      "latency_ms": 4